[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_62_External_Data_Corpus_Search.ipynb)

# Lesson 62 — External Data: Corpus-Level Semantic Search Over a Paper Collection

**Phase 6 · Lesson 7 of 10 (Building `paper-distiller` as a real OSS project)**

| # | Lesson | Status |
|---|--------|--------|
| 56 | Phase 6 Kickoff — project selection & scaffold | ✅ |
| 57 | Core Pipeline + Golden Eval Harness | ✅ |
| 58 | Typer CLI + PyPI Packaging | ✅ |
| 59 | Web API (FastAPI, async jobs, deploy) | ✅ |
| 60 | OSS Growth (README, badges, contributor funnel) | ✅ |
| 61 | agent-bench: Agent Benchmark Harness | ✅ |
| **62** | **External Data: corpus-level semantic search** | 👉 today |
| 63 | Safety & Guardrails | ⏭️ |
| 64 | Launch Day | ⏭️ |

### Where we left off

Every `paper_distiller` lesson so far has operated on **one paper at a time**: fetch → extract → digest. That's the right shape for "explain this paper to me." It is the *wrong* shape for "how do these 5 papers differ in how they handle X?" — a question that requires reasoning **across** documents, not within one.

Today we build the missing layer: a persistent, searchable **corpus** of papers, with semantic search and multi-document synthesis. This is the same retrieval machinery from L52 (**hybrid dense+BM25 search, cross-encoder re-ranking**) — but applied at the scale of an ever-growing external knowledge base instead of a single in-memory demo corpus, and backed by a real persistent vector store (ChromaDB) instead of an in-memory list.

### Today's concept: single-document extraction vs. corpus-level retrieval

| | Single-paper `/distill` (L56–59) | Corpus search + synthesis (today) |
|---|---|---|
| Input | One arXiv ID | A question, searched against N ingested papers |
| Retrieval | None — whole paper (via sections) goes to Claude | Semantic search picks the relevant chunks first |
| Output | One structured `PaperDigest` | A synthesized answer citing 1–N papers by ID |
| Storage | Stateless — nothing persists between calls | Persistent vector store — corpus grows over time |
| Cost driver | Tokens of one paper | Embedding cost (cheap) + tokens of top-k retrieved chunks (small) |
| Failure mode | Missed section, bad extraction | Missed *paper* (retrieval recall), hallucinated citation |

This is literally what separates a "PDF summarizer" from a "research assistant" — the latter needs memory across documents.


In [ ]:
# Setup — external data / corpus search dependencies
!pip install -q chromadb sentence-transformers rank-bm25 anthropic pydantic rich requests pdfplumber nest_asyncio

import os, glob, json, time, hashlib, textwrap
from dataclasses import dataclass, field
from typing import Optional

import nest_asyncio
nest_asyncio.apply()

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
console = Console()

# API key — Colab Secrets first, then env var, else graceful fallback (matches every prior lesson's pattern)
HAVE_API_KEY = False
ANTHROPIC_API_KEY = None
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    HAVE_API_KEY = bool(ANTHROPIC_API_KEY)
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
    HAVE_API_KEY = bool(ANTHROPIC_API_KEY)

if HAVE_API_KEY:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    console.print("[green]✓ Anthropic API key found — live synthesis cells will call Claude[/green]")
else:
    console.print("[yellow]⚠ No ANTHROPIC_API_KEY found — synthesis cells will use a deterministic offline fallback so the notebook still runs end-to-end[/yellow]")

CORPUS_DIR = "/content/paper_corpus_db"
os.makedirs(CORPUS_DIR, exist_ok=True)
print("Corpus persistence directory:", CORPUS_DIR)


## 1. Minimal FetchLayer (self-contained recreation)

Every lesson notebook must run standalone in a fresh Colab runtime, so we recreate the small FetchLayer piece from L56/L57 rather than assuming those files exist. In your real repo, this already lives in `paper_distiller/fetch.py` — you'd just import it.


In [ ]:
import re
import requests
import pdfplumber
import io

def parse_arxiv_id(raw: str) -> str:
    """Accepts a bare ID, an /abs/ URL, or a /pdf/ URL and returns the bare ID."""
    raw = raw.strip()
    m = re.search(r"(\d{4}\.\d{4,5})(v\d+)?", raw)
    if not m:
        raise ValueError(f"Could not parse an arXiv ID out of: {raw}")
    return m.group(1)

def fetch_metadata(arxiv_id: str) -> dict:
    url = f"http://export.arxiv.org/api/query?id_list={arxiv_id}"
    resp = requests.get(url, timeout=20)
    resp.raise_for_status()
    xml = resp.text
    title = re.search(r"<title>(.*?)</title>", xml, re.S)
    title = re.sub(r"\s+", " ", title.group(1)).strip() if title else arxiv_id
    # crude: skip the first <title> which is the API feed title
    titles = re.findall(r"<title>(.*?)</title>", xml, re.S)
    real_title = re.sub(r"\s+", " ", titles[1]).strip() if len(titles) > 1 else title
    summary = re.search(r"<summary>(.*?)</summary>", xml, re.S)
    abstract = re.sub(r"\s+", " ", summary.group(1)).strip() if summary else ""
    return {"arxiv_id": arxiv_id, "title": real_title, "abstract": abstract}

def fetch_pdf_text(arxiv_id: str, max_pages: int = 15) -> str:
    url = f"https://arxiv.org/pdf/{arxiv_id}"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    text_parts = []
    with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
        for page in pdf.pages[:max_pages]:
            t = page.extract_text() or ""
            text_parts.append(t)
    return "\n".join(text_parts)

# Smoke test on a well-known paper
meta = fetch_metadata("1706.03762")
console.print(Panel(f"[bold]{meta['title']}[/bold]\n\n{meta['abstract'][:300]}...", title=meta['arxiv_id']))


## 2. Chunking: from full text to searchable units

We reuse the **semantic chunking** idea from L52 (split on sentence-embedding discontinuities) but simplify to a robust, dependency-light **paragraph-window chunker** — sturdier across the wildly different section conventions we've already seen fail in L57 (Pitfall #1: section blindness). Each chunk carries its `arxiv_id` and a `chunk_index` so retrieval results can always be traced back to a specific paper and location.


In [ ]:
from typing import List
from pydantic import BaseModel

class Chunk(BaseModel):
    arxiv_id: str
    chunk_index: int
    text: str

def chunk_text(arxiv_id: str, text: str, target_chars: int = 900, overlap_chars: int = 150) -> List[Chunk]:
    """Paragraph-aware sliding window chunker.

    Splits on blank lines first (paragraphs), then packs paragraphs into
    ~target_chars windows with a small overlap so a fact split across a
    paragraph boundary is still retrievable from at least one chunk.
    """
    paras = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks, buf = [], ""
    for p in paras:
        if len(buf) + len(p) + 1 <= target_chars:
            buf = (buf + "\n" + p).strip()
        else:
            if buf:
                chunks.append(buf)
            # start new buffer with overlap tail of previous buffer
            tail = buf[-overlap_chars:] if len(buf) > overlap_chars else buf
            buf = (tail + "\n" + p).strip()
    if buf:
        chunks.append(buf)
    return [Chunk(arxiv_id=arxiv_id, chunk_index=i, text=c) for i, c in enumerate(chunks)]

sample_chunks = chunk_text(meta["arxiv_id"], meta["abstract"] + "\n\n" + fetch_pdf_text(meta["arxiv_id"], max_pages=3))
print(f"Produced {len(sample_chunks)} chunks from the first 3 pages")
print("--- sample chunk 0 ---")
print(sample_chunks[0].text[:400])


## 3. The `CorpusStore`: a persistent vector store, not an in-memory list

This is the core upgrade over L52. There, `hybrid_search()` ran against a 20-document Python list that vanished when the runtime restarted. Here, `CorpusStore` wraps a **ChromaDB `PersistentClient`** — embeddings and metadata are written to disk (`/content/paper_corpus_db`), so the corpus survives across notebook restarts, and you can keep adding papers to it over weeks without re-embedding what's already there (dedup check on `arxiv_id`).


In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

class CorpusStore:
    """Persistent, incrementally-growable semantic index over paper chunks."""

    def __init__(self, persist_dir: str = CORPUS_DIR, collection_name: str = "papers"):
        self.client = chromadb.PersistentClient(path=persist_dir)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"embedding_model": EMBED_MODEL_NAME},
        )
        self.embedder = SentenceTransformer(EMBED_MODEL_NAME)

    def has_paper(self, arxiv_id: str) -> bool:
        existing = self.collection.get(where={"arxiv_id": arxiv_id}, limit=1)
        return len(existing["ids"]) > 0

    def ingest(self, arxiv_id: str, title: str, chunks: List[Chunk]) -> int:
        """Embeds and stores chunks. Skips (dedups) if arxiv_id already present."""
        if self.has_paper(arxiv_id):
            return 0
        texts = [c.text for c in chunks]
        embeddings = self.embedder.encode(texts, show_progress_bar=False).tolist()
        ids = [f"{arxiv_id}::{c.chunk_index}" for c in chunks]
        metadatas = [{"arxiv_id": arxiv_id, "title": title, "chunk_index": c.chunk_index} for c in chunks]
        self.collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)
        return len(chunks)

    def search(self, query: str, top_k: int = 5, arxiv_id_filter: Optional[str] = None) -> List[dict]:
        query_emb = self.embedder.encode([query]).tolist()
        where = {"arxiv_id": arxiv_id_filter} if arxiv_id_filter else None
        results = self.collection.query(query_embeddings=query_emb, n_results=top_k, where=where)
        hits = []
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
            hits.append({"text": doc, "arxiv_id": meta["arxiv_id"], "title": meta["title"],
                         "chunk_index": meta["chunk_index"], "distance": dist})
        return hits

    def stats(self) -> dict:
        all_meta = self.collection.get()["metadatas"]
        papers = {}
        for m in all_meta:
            papers.setdefault(m["arxiv_id"], {"title": m["title"], "chunks": 0})
            papers[m["arxiv_id"]]["chunks"] += 1
        return {"total_chunks": len(all_meta), "papers": papers}

store = CorpusStore()
print("CorpusStore ready. Current stats:", store.stats())


## 4. Ingesting a small corpus

We ingest a handful of landmark papers — the same trio used for the golden eval set in L57, plus two more — so the corpus has enough breadth for cross-paper questions to be meaningful. `ingest()` is idempotent: re-running this cell after the first pass costs nothing (dedup check via `has_paper`).


In [ ]:
SEED_PAPERS = [
    "1706.03762",  # Attention Is All You Need
    "1810.04805",  # BERT
    "2106.09685",  # LoRA
    "2005.14165",  # GPT-3
    "2203.02155",  # InstructGPT
]

ingest_report = []
for arxiv_id in SEED_PAPERS:
    if store.has_paper(arxiv_id):
        ingest_report.append((arxiv_id, "skipped (already in corpus)", 0))
        continue
    m = fetch_metadata(arxiv_id)
    full_text = m["abstract"] + "\n\n" + fetch_pdf_text(arxiv_id, max_pages=12)
    chunks = chunk_text(arxiv_id, full_text)
    n_added = store.ingest(arxiv_id, m["title"], chunks)
    ingest_report.append((arxiv_id, m["title"][:60], n_added))
    time.sleep(1)  # be polite to arXiv

table = Table(title="Ingest report")
table.add_column("arXiv ID"); table.add_column("Title / status"); table.add_column("Chunks added", justify="right")
for r in ingest_report:
    table.add_row(*[str(x) for x in r])
console.print(table)
console.print(store.stats())


## 5. Semantic search across the whole corpus

This is the payoff: one query, ranked results **from any paper in the corpus**, each result carrying its source `arxiv_id` so we always know where a fact came from. Compare this to L52's `hybrid_search()` — same idea, but now querying a disk-backed index that already has real papers in it instead of 20 synthetic demo docs.


In [ ]:
def print_search_results(query: str, hits: List[dict]):
    table = Table(title=f'Search: "{query}"')
    table.add_column("Paper"); table.add_column("arXiv ID"); table.add_column("Score", justify="right"); table.add_column("Snippet")
    for h in hits:
        # Dense hits carry "distance" (lower=better); BM25/fused hits may instead
        # carry "rank_score" (higher=better) or no score at all -- display whichever
        # is present rather than assuming one fixed schema across result sources.
        if "distance" in h:
            score_str = f'dist={h["distance"]:.3f}'
        elif "rank_score" in h:
            score_str = f'bm25={h["rank_score"]:.3f}'
        else:
            score_str = "-"
        table.add_row(h["title"][:35], h["arxiv_id"], score_str, h["text"][:90].replace("\n", " ") + "…")
    console.print(table)

hits = store.search("how does the model decide what to attend to", top_k=5)
print_search_results("how does the model decide what to attend to", hits)

hits2 = store.search("efficient fine-tuning with low-rank adapters", top_k=5)
print_search_results("efficient fine-tuning with low-rank adapters", hits2)


## 6. Hybrid re-ranking (carrying L52 forward)

Pure dense search misses exact-keyword matches (model names, metric names). We add the same **BM25 + Reciprocal Rank Fusion** approach from L52, now operating over the full corpus's chunk texts, pulled live from the Chroma collection rather than a fixed in-memory list.


In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25_index(store: CorpusStore):
    all_data = store.collection.get()
    docs = all_data["documents"]
    metas = all_data["metadatas"]
    ids = all_data["ids"]
    tokenized = [d.lower().split() for d in docs]
    bm25 = BM25Okapi(tokenized)
    return bm25, docs, metas, ids

def bm25_search(store: CorpusStore, query: str, top_k: int = 10):
    bm25, docs, metas, ids = build_bm25_index(store)
    scores = bm25.get_scores(query.lower().split())
    ranked = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"text": docs[i], "arxiv_id": metas[i]["arxiv_id"], "title": metas[i]["title"], "rank_score": scores[i]} for i in ranked]

def rrf_fuse(dense_hits: List[dict], sparse_hits: List[dict], k: int = 60, top_k: int = 5) -> List[dict]:
    scores = {}
    items = {}
    for rank, h in enumerate(dense_hits):
        key = (h["arxiv_id"], h["text"][:50])
        scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
        items[key] = h
    for rank, h in enumerate(sparse_hits):
        key = (h["arxiv_id"], h["text"][:50])
        scores[key] = scores.get(key, 0) + 1.0 / (k + rank + 1)
        items[key] = h
    fused_keys = sorted(scores, key=lambda k_: scores[k_], reverse=True)[:top_k]
    return [items[k_] for k_ in fused_keys]

def hybrid_corpus_search(store: CorpusStore, query: str, top_k: int = 5):
    dense_hits = store.search(query, top_k=10)
    sparse_hits = bm25_search(store, query, top_k=10)
    return rrf_fuse(dense_hits, sparse_hits, top_k=top_k)

fused = hybrid_corpus_search(store, "reinforcement learning from human feedback")
print_search_results("reinforcement learning from human feedback (hybrid)", fused)


## 7. Corpus-level synthesis: `ask_corpus()`

This is the piece that has no equivalent in L56–59. Instead of extracting a digest from one paper, we retrieve the top chunks **across the whole corpus** and ask Claude to synthesize an answer that explicitly cites which paper(s) each claim comes from — the multi-document RAG pattern. Every claim must be traceable to a `[arxiv_id]` tag; if the retrieved context doesn't support an answer, the model is instructed to say so rather than fill the gap from parametric memory.


In [ ]:
SYNTH_SYSTEM = """You are a research assistant answering questions using ONLY the provided
paper excerpts. Rules:
1. Every factual claim must end with a citation tag like [1706.03762].
2. If the excerpts don't contain enough information to answer, say so explicitly —
   do not fill gaps from general knowledge.
3. If multiple papers take different approaches, contrast them explicitly.
4. Keep the answer to 150-250 words."""

def _format_context(hits: List[dict]) -> str:
    blocks = []
    for h in hits:
        blocks.append(f"[{h['arxiv_id']}] ({h['title']}):\n{h['text']}")
    return "\n\n---\n\n".join(blocks)

def _offline_fallback_answer(question: str, hits: List[dict]) -> str:
    """Deterministic stand-in used when no API key is present, so the notebook
    still demonstrates the full ask_corpus() contract end-to-end for free."""
    cited = sorted(set(h["arxiv_id"] for h in hits))
    lines = [f"[offline fallback — no live Claude call] Retrieved {len(hits)} chunks from {len(cited)} paper(s): {', '.join(cited)}."]
    for h in hits[:3]:
        lines.append(f"- From [{h['arxiv_id']}]: \"{h['text'][:120].strip()}...\"")
    lines.append("(Set ANTHROPIC_API_KEY to see a real synthesized, cited answer here.)")
    return "\n".join(lines)

def ask_corpus(store: CorpusStore, question: str, top_k: int = 6) -> dict:
    hits = hybrid_corpus_search(store, question, top_k=top_k)
    if not hits:
        return {"question": question, "answer": "No relevant chunks found in corpus.", "sources": []}

    if HAVE_API_KEY:
        context = _format_context(hits)
        resp = client.messages.create(
            model="claude-sonnet-4-5",
            max_tokens=500,
            system=SYNTH_SYSTEM,
            messages=[{"role": "user", "content": f"Question: {question}\n\nExcerpts:\n\n{context}"}],
        )
        answer = resp.content[0].text
    else:
        answer = _offline_fallback_answer(question, hits)

    return {"question": question, "answer": answer, "sources": sorted(set(h["arxiv_id"] for h in hits))}

result = ask_corpus(store, "What are two different approaches to adapting a large pretrained model to a new task, and how do they compare in cost?")
console.print(Panel(result["answer"], title=f"Q: {result['question']}", subtitle=f"Sources: {', '.join(result['sources'])}"))


## 8. Persistence check — does the corpus actually survive a restart?

`chromadb.PersistentClient` writes to disk on every `.add()` call. To prove this isn't just an in-memory illusion, we drop our Python reference to `store`, create a **brand-new `CorpusStore` instance** pointing at the same directory, and confirm the data is still there without re-ingesting anything.


In [ ]:
del store  # simulate losing the in-memory object (e.g., notebook restart)

store2 = CorpusStore()  # fresh instance, same persist_dir
stats_after_reload = store2.stats()
print("Stats after simulated restart (new CorpusStore instance, same disk path):")
print(json.dumps(stats_after_reload, indent=2)[:800])

assert stats_after_reload["total_chunks"] > 0, "Persistence failed — corpus should survive a fresh client instance"
assert len(stats_after_reload["papers"]) == len(SEED_PAPERS) or len(stats_after_reload["papers"]) > 0
print(f"\n✓ Persistence confirmed: {stats_after_reload['total_chunks']} chunks across {len(stats_after_reload['papers'])} papers survived reload.")

store = store2  # continue using the reloaded instance below


## 9. Wiring into `paper_distiller` as a real module

We write this as `paper_distiller/corpus/store.py` — a new subpackage alongside `fetch.py`, `extract.py`, `codegen.py` from L56, and `section_detector.py`, `batch.py` from L57. It's designed to be imported by the L58 Typer CLI (`paper-distiller corpus add <id>` / `corpus ask "<question>"`) and the L59 FastAPI service (`POST /corpus/ask`) without modification — same pattern as every prior module.


In [ ]:
import os
os.makedirs("/content/paper_distiller/corpus", exist_ok=True)

with open("/content/paper_distiller/corpus/__init__.py", "w") as f:
    f.write("")

Q3 = chr(34) * 3  # a triple-double-quote token, built at runtime to avoid nesting issues in this generator script

store_py_lines = [
    Q3 + "Persistent corpus-level semantic search over ingested papers.",
    "",
    "New in Phase 6 Lesson 62: unlike fetch.py/extract.py which operate on a",
    "single paper per call, this module maintains a growable, disk-persisted",
    "index across many papers, enabling cross-paper questions.",
    Q3,
    "import re",
    "from typing import List, Optional",
    "from pydantic import BaseModel",
    "",
    "",
    "class Chunk(BaseModel):",
    "    arxiv_id: str",
    "    chunk_index: int",
    "    text: str",
    "",
    "",
    "def chunk_text(arxiv_id: str, text: str, target_chars: int = 900, overlap_chars: int = 150) -> List[Chunk]:",
    '    paras = [p.strip() for p in re.split(r"\\n\\s*\\n", text) if p.strip()]',
    '    chunks, buf = [], ""',
    "    for p in paras:",
    "        if len(buf) + len(p) + 1 <= target_chars:",
    '            buf = (buf + "\\n" + p).strip()',
    "        else:",
    "            if buf:",
    "                chunks.append(buf)",
    "            tail = buf[-overlap_chars:] if len(buf) > overlap_chars else buf",
    '            buf = (tail + "\\n" + p).strip()',
    "    if buf:",
    "        chunks.append(buf)",
    "    return [Chunk(arxiv_id=arxiv_id, chunk_index=i, text=c) for i, c in enumerate(chunks)]",
    "",
    "",
    "class CorpusStore:",
    "    # Persistent, incrementally-growable semantic index over paper chunks.",
    "",
    '    def __init__(self, persist_dir: str = "./paper_corpus_db", collection_name: str = "papers"):',
    "        import chromadb",
    "        from sentence_transformers import SentenceTransformer",
    "        self.client = chromadb.PersistentClient(path=persist_dir)",
    "        self.collection = self.client.get_or_create_collection(name=collection_name)",
    '        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")',
    "",
    "    def has_paper(self, arxiv_id: str) -> bool:",
    '        existing = self.collection.get(where={"arxiv_id": arxiv_id}, limit=1)',
    '        return len(existing["ids"]) > 0',
    "",
    "    def ingest(self, arxiv_id: str, title: str, chunks: List[Chunk]) -> int:",
    "        if self.has_paper(arxiv_id):",
    "            return 0",
    "        texts = [c.text for c in chunks]",
    "        embeddings = self.embedder.encode(texts, show_progress_bar=False).tolist()",
    '        ids = [f"{arxiv_id}::{c.chunk_index}" for c in chunks]',
    '        metadatas = [{"arxiv_id": arxiv_id, "title": title, "chunk_index": c.chunk_index} for c in chunks]',
    "        self.collection.add(ids=ids, embeddings=embeddings, documents=texts, metadatas=metadatas)",
    "        return len(chunks)",
    "",
    "    def search(self, query: str, top_k: int = 5, arxiv_id_filter: Optional[str] = None) -> List[dict]:",
    "        query_emb = self.embedder.encode([query]).tolist()",
    '        where = {"arxiv_id": arxiv_id_filter} if arxiv_id_filter else None',
    "        results = self.collection.query(query_embeddings=query_emb, n_results=top_k, where=where)",
    "        hits = []",
    '        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):',
    '            hits.append({"text": doc, "arxiv_id": meta["arxiv_id"], "title": meta["title"],',
    '                         "chunk_index": meta["chunk_index"], "distance": dist})',
    "        return hits",
    "",
    "    def stats(self) -> dict:",
    '        all_meta = self.collection.get()["metadatas"]',
    "        papers = {}",
    "        for m in all_meta:",
    '            papers.setdefault(m["arxiv_id"], {"title": m["title"], "chunks": 0})',
    '            papers[m["arxiv_id"]]["chunks"] += 1',
    '        return {"total_chunks": len(all_meta), "papers": papers}',
    "",
]

with open("/content/paper_distiller/corpus/store.py", "w") as f:
    f.write("\n".join(store_py_lines))

print("Wrote paper_distiller/corpus/__init__.py and paper_distiller/corpus/store.py")
for fn in sorted(os.listdir("/content/paper_distiller/corpus")):
    print(" -", fn)


### CLI extension sketch (Typer subcommand group, from L58)

```python
# paper_distiller/cli.py — add alongside cmd_distill / cmd_batch / cmd_eval
corpus_app = typer.Typer(help="Manage the cross-paper semantic search corpus")
app.add_typer(corpus_app, name="corpus")

@corpus_app.command("add")
def corpus_add(arxiv_id: str):
    """Ingest a paper into the persistent corpus (idempotent)."""
    ...

@corpus_app.command("ask")
def corpus_ask(question: str, top_k: int = 6):
    """Ask a question synthesized across all ingested papers, with citations."""
    ...
```

And the matching FastAPI route sketch (from L59): `POST /corpus/ask {"question": "...", "top_k": 6}` → same `ask_corpus()` call, wrapped in the existing auth + rate-limit middleware stack.


## 10. Pitfalls

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | Re-embedding the whole corpus on every run | `has_paper()` dedup exists for a reason — without it, ingesting the same 5 papers daily burns embedding compute for nothing |
| 2 | No persistence directory (using `chromadb.Client()` instead of `PersistentClient`) | Silent in-memory mode — corpus vanishes on restart, looks like data loss the next day |
| 3 | Mixing embedding models across ingests | If you swap `all-MiniLM-L6-v2` for a different model later, old vectors are in a different space — searches silently degrade; store the model name in collection metadata (we do) and re-embed everything on a model change |
| 4 | No source attribution in synthesis | An answer with no `[arxiv_id]` citations is unverifiable and impossible to fact-check against the source |
| 5 | Hallucinated citations | The model can still cite a paper ID that wasn't actually in the retrieved context — always validate cited IDs are a subset of `sources` before trusting them in production |
| 6 | Chunks too large | A 3000-char chunk dilutes the embedding — a specific fact gets buried in a big semantic average and stops being retrievable; we use ~900 chars for a reason |
| 7 | Chunks too small with no overlap | Cuts a sentence in half at a chunk boundary — the overlap_chars tail exists to reduce (not eliminate) this |
| 8 | Unbounded corpus growth with no pruning | Nothing here caps corpus size or evicts stale papers — fine for a hobby project, not fine for production without a retention policy |
| 9 | No retrieval eval for the corpus itself | L52 had a golden retrieval eval (Recall@k/MRR/NDCG) — this lesson doesn't add one; do that before trusting `ask_corpus()` on real questions |
| 10 | Treating `ask_corpus()` like a chatbot with unlimited knowledge | It only knows what's been ingested — a paper not yet in the corpus simply won't be found, and the system prompt is designed to say "not enough information" rather than guess |


In [ ]:
# Verification checklist
checks = []

checks.append(("CorpusStore importable and instantiable", store is not None))
checks.append(("Corpus has papers ingested", len(store.stats()["papers"]) > 0))
checks.append(("Dense search returns results", len(hits) > 0))
checks.append(("Hybrid (RRF-fused) search returns results", len(fused) > 0))
checks.append(("ask_corpus() returns an answer + sources", bool(result["answer"]) and len(result["sources"]) > 0))
checks.append(("Persistence survives a fresh CorpusStore instance", stats_after_reload["total_chunks"] > 0))
checks.append(("corpus/store.py written to disk", os.path.exists("/content/paper_distiller/corpus/store.py")))
checks.append(("corpus/__init__.py written to disk", os.path.exists("/content/paper_distiller/corpus/__init__.py")))

table = Table(title="Lesson 62 verification")
table.add_column("Check"); table.add_column("Result")
all_pass = True
for name, ok in checks:
    table.add_row(name, "✅ PASS" if ok else "❌ FAIL")
    all_pass = all_pass and ok
console.print(table)
print("\nALL CHECKS PASSED" if all_pass else "\nSOME CHECKS FAILED — review above")
assert all_pass


## Summary

| Concept | What you built |
|---|---|
| Single-paper vs. corpus-level retrieval | Understood *why* `/distill` and `ask_corpus()` are structurally different problems |
| Paragraph-window chunking | A robust chunker that survives varied section conventions (unlike L57's section-blind text) |
| `CorpusStore` (ChromaDB `PersistentClient`) | A disk-persisted, incrementally-growable semantic index — corpus survives restarts |
| Idempotent ingestion | `has_paper()` dedup so re-running ingestion is free |
| Dense + BM25 + RRF hybrid search | Carried L52's retrieval quality techniques forward onto a real, growing corpus |
| `ask_corpus()` multi-document synthesis | Cited, cross-paper answers with an explicit "don't know" escape hatch |
| Persistence verification | Proved data survives a fresh client instance, not just an in-memory illusion |
| Module wiring | `paper_distiller/corpus/store.py` ready for the L58 CLI and L59 API to call directly |

### Homework

1. Add a **retrieval eval harness** for the corpus (golden query → expected arxiv_id set) and compute Recall@5/MRR, following the L52 pattern.
2. Add a citation-validator: after `ask_corpus()` returns, regex-extract all `[arxiv_id]` tags in the answer and assert each one is in `result["sources"]`; log/flag any that aren't.
3. Add a `corpus.evict(arxiv_id)` method and a retention policy (e.g., max 500 papers, evict oldest by ingestion timestamp).
4. Swap `all-MiniLM-L6-v2` for a larger embedding model, re-embed the whole corpus, and A/B the search quality on 5 hand-written queries.
5. Wire `corpus add` / `corpus ask` into the real Typer CLI (L58) and the FastAPI service (L59) in your actual `paper-distiller` repo, then commit + push.

### Next lesson (63): Safety & Guardrails

We'll add prompt-injection defenses (a malicious paper abstract trying to hijack `extract_digest()`), input/output validation, and a red-team eval suite — using the same benchmark harness pattern from L61 to score *safety* instead of *capability*.
